In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs
fs.CELL_COLORS = fs.HUMAN_CELL_COLORS

ad.settings.zarr_write_format = 2

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "01_export_and_annotate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_human_preprocessed.zarr")
adata

In [ ]:
mix = pd.crosstab(adata.obs["leiden"], adata.obs["roi_name"], normalize="index").round(3)
mix["cells"] = adata.obs["leiden"].value_counts()
mix.to_csv(OUT_DIR / "leiden_roi_fraction.csv")
mix.loc[mix.drop(columns="cells").max(axis=1).sort_values(ascending=False).index]

In [ ]:
print("Total cells:", adata.n_obs)
adata.obs.groupby("roi_name", observed=True)["total_counts"].agg(
    n_cells="size", n_transcripts="sum", median_per_cell="median")

In [ ]:
fig, ax = plt.subplots(figsize=(fs.COL2, 40 * fs.MM), layout="constrained")
sc.pl.violin(adata, "total_counts", groupby="leiden", log=True, stripplot=False, ax=ax, show=False)
for violin in ax.collections:
    violin.set_facecolor(fs.LIGHTGREY)
    violin.set_edgecolor(fs.GREY)
    violin.set_linewidth(0.3)
ax.set_xlabel("Cluster")
ax.set_ylabel("Transcripts per cell")
ax.tick_params(axis="x", labelrotation=0)
plt.show()

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True, n_genes=100)
deg = sc.get.rank_genes_groups_df(adata, group=None)
deg.to_csv(OUT_DIR / "rank_genes_leiden.csv", index=False)
deg.groupby("group", observed=True).head(3)

In [ ]:
top10 = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top10.index = range(1, 11)
top10.to_csv(OUT_DIR / "rank_genes_leiden_top10.csv")
top10

In [ ]:
markers = {
    "Myofiber":         ["MYH2", "MYOT"],
    "Fibroblast":       ["PDGFRA", "DPT"],
    "MuSC":             ["PAX7"],
    "Perineurial cell": ["NGFR", "CLDN1"],
    "Schwann cell":     ["MPZ", "SOX10"],
    "Adipocyte":        ["ADIPOQ", "PLIN1"],
    "Endothelial cell": ["PECAM1"],
    "Arterial EC":      ["GJA5", "EFNB2", "PODXL"],
    "EC":               ["NOTCH4", "ITGA6"],
    "Venous EC":        ["CCL14", "PLVAP", "SELP"],
    "Lymphatic EC":     ["FLT4", "PROX1", "PDPN"],
    "Arterial SMC":     ["MYLK", "CSRP1", "PLN"],
    "Venous SMC":       ["SYNPO2", "ADAMTS4", "STEAP4"],
    "Pericyte":         ["RGS5", "NOTCH3"],
    "Macrophage":       ["CD163", "MRC1"],
    "Neutrophil":       ["CSF3R", "FPR1"],
    "T cell":           ["CD3E", "CD8A"],
    "Plasma cell":      ["MZB1", "XBP1", "CD79A"],
    "Mast cell":        ["KIT"],
}

In [ ]:
def cluster_dotplot(adata, markers, cluster, label=None, block=None, height=100 * fs.MM, width=fs.COL2, brackets=True,
                    dot_max=1, legend_width=1.5):
    obs = adata.obs
    names = list(markers)
    genes = {ct: [g for g in gs if g in adata.var_names] for ct, gs in markers.items()}
    genes = {ct: gs for ct, gs in genes.items() if gs}
    flat = [g for gs in genes.values() for g in gs]

    table = pd.DataFrame({"cluster": obs[cluster].astype(str)})
    table["block"] = obs[block].astype(str) if block is not None else ""
    table = table.drop_duplicates("cluster").set_index("cluster", drop=False).rename_axis(None)
    table["rank"] = table["block"].map(lambda b: names.index(b) if b in names else len(names))
    table["prefix"] = table["cluster"].str.replace(r"\d+$", "", regex=True)
    table["num"] = pd.to_numeric(table["cluster"].str.extract(r"(\d+)$")[0], errors="coerce")
    if label is None:
        x = adata[:, flat].layers["log1p_norm"]
        x = x.toarray() if hasattr(x, "toarray") else np.asarray(x)
        mean = pd.DataFrame(x, columns=flat).groupby(obs[cluster].astype(str).to_numpy()).mean()
        scaled = (mean - mean.min()) / (mean.max() - mean.min())
        score = pd.DataFrame({ct: scaled[gs].mean(axis=1) for ct, gs in genes.items()})
        table["rank"] = score.loc[table.index].to_numpy().argmax(axis=1)
        blocks = list(genes.values())
        table["top_gene"] = [flat.index(scaled.loc[c, blocks[r]].idxmax()) for c, r in zip(table.index, table["rank"])]
        table = table.sort_values(["rank", "top_gene", "prefix", "num", "cluster"])
    else:
        table = table.sort_values(["rank", "prefix", "num", "cluster"])
    rows = list(table["cluster"])

    key = "_cluster_label"
    obs[key] = pd.Categorical(obs[cluster].astype(str).map(dict(zip(table["cluster"], rows))), categories=rows)
    try:
        dp = sc.pl.dotplot(adata, genes if brackets else flat, groupby=key, layer="log1p_norm", standard_scale="var",
                           dot_min=0, dot_max=dot_max,
                           figsize=(width, height), show=False, return_fig=True)
        dp.style(largest_dot=40, dot_edge_lw=0.2)
        dp.legend(width=legend_width, size_title="% cells", colorbar_title="Scaled\nexpression")
        dp.make_figure()
    finally:
        del obs[key]

    ax = dp.get_axes()["mainplot_ax"]
    for t in ax.get_xticklabels():
        t.set_fontstyle("italic")
    ax.set_ylabel("Cluster" if label is None else "Cell type")
    return dp

dp = cluster_dotplot(adata, markers, "leiden", height=85 * fs.MM)
fs.save(dp.fig, OUT_DIR / "dotplot_markers")
plt.show()

markers_large = {
    "Myofiber":           ["MYH2", "MYOT"],
    "Fibroblast":         ["PDGFRA", "DPT"],
    "MuSC":               ["PAX7"],
    "Perineurial cell":   ["NGFR", "CLDN1"],
    "Schwann cell":       ["MPZ", "SOX10"],
    "Adipocyte":          ["ADIPOQ", "PLIN1"],
    "Endothelial cell":   ["PECAM1", "CDH5"],
    "Smooth muscle cell": ["MYLK", "NOTCH3"],
    "Pericyte":           ["RGS5"],
    "Immune cell":        ["CD163", "CSF3R", "CD3E", "MZB1", "KIT"],
}
dp = cluster_dotplot(adata, markers_large, "leiden", height=85 * fs.MM, width=fs.COL1 * 1.5)
fs.save(dp.fig, OUT_DIR / "dotplot_markers_large")
plt.show()

In [ ]:
subtype_clusters = {
    "Myofiber":         ["0", "4", "7", "9", "13", "18"],
    "Fibroblast":       ["2", "3", "14", "25"],
    "MuSC":             ["16"],
    "Perineurial cell": ["19"],
    "Schwann cell":     ["21"],
    "Adipocyte":        ["23"],
    "Arterial EC":      ["20"],
    "EC":               ["5"],
    "Venous EC":        ["6"],
    "Lymphatic EC":     ["26"],
    "Arterial SMC":     ["12"],
    "Venous SMC":       ["17"],
    "Pericyte":         ["10"],
    "Macrophage":       ["1"],
    "Neutrophil":       ["11"],
    "T cell":           ["15"],
    "Plasma cell":      ["22"],
    "Mast cell":        ["24"],
    "Unassigned":       ["8"],
}
adata.obs["cell_subtype"] = ""
for subtype, clusters in subtype_clusters.items():
    adata.obs.loc[adata.obs["leiden"].isin(clusters), "cell_subtype"] = subtype
adata.obs["cell_subtype"] = pd.Categorical(adata.obs["cell_subtype"], categories=list(subtype_clusters))

largetype_subtypes = {
    "Myofiber":           ["Myofiber"],
    "Fibroblast":         ["Fibroblast"],
    "MuSC":               ["MuSC"],
    "Perineurial cell":   ["Perineurial cell"],
    "Schwann cell":       ["Schwann cell"],
    "Adipocyte":          ["Adipocyte"],
    "Endothelial cell":   ["Arterial EC", "EC", "Venous EC", "Lymphatic EC"],
    "Smooth muscle cell": ["Arterial SMC", "Venous SMC"],
    "Pericyte":           ["Pericyte"],
    "Immune cell":        ["Macrophage", "Neutrophil", "T cell", "Plasma cell", "Mast cell"],
    "Unassigned":         ["Unassigned"],
}
adata.obs["cell_largetype"] = ""
for largetype, subtypes in largetype_subtypes.items():
    adata.obs.loc[adata.obs["cell_subtype"].isin(subtypes), "cell_largetype"] = largetype
adata.obs["cell_largetype"] = pd.Categorical(adata.obs["cell_largetype"], categories=list(largetype_subtypes))

pd.crosstab(adata.obs["cell_subtype"], adata.obs["roi_name"])

In [ ]:
def umap(ax, xy, labels, size=0.3):
    labels = pd.Categorical(labels).remove_unused_categories()
    cats = labels.categories
    order = np.random.default_rng(0).permutation(len(xy))
    colors = np.array(fs.palette(labels.astype(str)))
    ax.scatter(xy[order, 0], xy[order, 1], c=colors[order], s=size, lw=0, rasterized=True)
    ax.set_aspect("equal")
    ax.set_axis_off()

    handles = [mpl.lines.Line2D([], [], ls="", marker="o", ms=3, mew=0, color=c) for c in fs.palette(cats)]
    ax.legend(handles, cats, loc="center left", bbox_to_anchor=(1.0, 0.5), borderaxespad=0, handletextpad=0.1,
              labelspacing=0.25)

    span = np.ptp(xy, axis=0).max()
    x0, y0 = xy.min(axis=0) - 0.03 * span
    arrow = dict(arrowstyle="-|>", lw=0.5, color=fs.BLACK, mutation_scale=4, shrinkA=0, shrinkB=0)
    ax.update_datalim([(x0, y0)])
    ax.annotate("", (x0 + 0.15 * span, y0), (x0, y0), arrowprops=arrow)
    ax.annotate("", (x0, y0 + 0.15 * span), (x0, y0), arrowprops=arrow)
    ax.annotate("UMAP1", (x0 + 0.075 * span, y0), xytext=(0, -2), textcoords="offset points", fontsize=5,
                ha="center", va="top")
    ax.annotate("UMAP2", (x0, y0 + 0.075 * span), xytext=(-2, 0), textcoords="offset points", fontsize=5,
                ha="right", va="center", rotation=90)
    return ax

def save_umap(color, name):
    fig, ax = plt.subplots(figsize=(1.2 * fs.COL1, 0.85 * fs.COL1), layout="constrained")
    umap(ax, adata.obsm["X_umap"], adata.obs[color], size=0.5)
    fs.save(fig, OUT_DIR / name)
    plt.show()

save_umap("cell_subtype", "umap_cell_subtype")
save_umap("cell_largetype", "umap_cell_largetype")

In [ ]:
def composition(obs, group, stack, groups, title=None, width=fs.COL2, height=55 * fs.MM, panels=("count", "fraction"),
                rows=None, row_labels=None, tick_label=None):
    ct = pd.crosstab(obs[group], obs[stack]).reindex(index=groups, fill_value=0)
    ct = ct.loc[:, ct.sum() > 0]
    frac = ct.div(ct.sum(axis=1), axis=0) * 100
    spec = {"count": (ct, "Number of cells"), "fraction": (frac, "Proportion of cells (%)")}
    rows = rows or [groups]
    fig, axes = plt.subplots(len(rows), len(panels), figsize=(width, height), layout="constrained", squeeze=False,
                             sharey="col")
    for r, row in enumerate(rows):
        for ax, panel in zip(axes[r], panels):
            table, ylabel = spec[panel]
            table = table.loc[row]
            xs = np.arange(len(table))
            bottom = np.zeros(len(table))
            for c, color in zip(table.columns, fs.palette(table.columns)):
                ax.bar(xs, table[c], bottom=bottom, width=0.75, color=color, edgecolor="white", linewidth=0.25, label=c)
                bottom += table[c].to_numpy(dtype=float)
            if tick_label is None:
                ax.set_xticks(xs, [fs.label(g) for g in table.index], rotation=45, ha="right", rotation_mode="anchor")
            else:
                ax.set_xticks(xs, [tick_label(g) for g in table.index])
            if row_labels is not None:
                ax.set_title(row_labels[r])
            ax.set_ylabel(ylabel)
            if panel == "count":
                ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
            else:
                ax.set_ylim(0, 100)
    handles, names = axes[0, -1].get_legend_handles_labels()
    fig.legend(handles[::-1], names[::-1], title=title, loc="outside right center", alignment="left")
    return fig, ct

ROIS = ["SF1", "SF2", "Xray_Under", "Xray_Late", "Cion_Late"]

def save_composition(cells, stack, name, title=None):
    fig, ct = composition(cells, "roi_name", stack, ROIS, title=title, width=0.8 * fs.COL2)
    fs.save(fig, OUT_DIR / f"composition_{name}")
    ct.to_csv(OUT_DIR / f"composition_{name}.csv")
    plt.show()

obs = adata.obs
save_composition(obs, "cell_largetype", "cell_largetype")
save_composition(obs, "cell_subtype", "cell_subtype")
save_composition(obs[obs["cell_largetype"] == "Immune cell"], "cell_subtype", "immune", "Immune cells")

In [ ]:
for key in [key for key in adata.uns if key.endswith("_colors")]:
    del adata.uns[key]
adata.write_zarr(BASE / "data" / "integrated_human_annotated.zarr")